## Analysis Notebook for AI Sycophancy and Prompt Similarity

Given $N$ prompt items, and model `m` in the set of `M` models for each item $i \in \set{1, ..., N}$:
- The Reddit AITA prompt = $p_i$
- The LLM's response to that prompt = $r_i^m$
- Model `m`'s syccophancy label = $y_i^m\in\set{0, 1}$
- The most upvoted crowd response to that prompt = $r_i^C$
- The SBERT embedding of prompt $i$ is $e_i^p$

Asssumptions:
- $r_i^C$ is treated as a ground truth human response for prompt $i$
- Divergence from $r_i^C$ is a syccophancy flag


**Research Questions:**

1. Are prompts whose LLM responses are labeled sycophantic more semantically similar to one another than prompts whose LLM responses are labeled non-sycophantic?
    1. Are there perceptible clusters in sycophancy-generated prompts, respective to each model?
2. Are LLM responses labeled sycophantic less semantically similar to the most upvoted crowd response than non-sycophantic LLM responses?


## Config and Table Loads

In [1]:
from pathlib import Path
from typing import Sequence

import pandas as pd
from sentence_transformers import SentenceTransformer

import torch

import numpy as np

from db.crud import get_all_responses, get_all_prompts, get_all_system_prompts

SENTENCE_TRANSFORMER_MODEL_FAST = "all-MiniLM-L6-v2"
SENTENCE_TRANSFORMER_MODEL_POWERFUL = "sentence-transformers/all-mpnet-base-v2"

# Switch to powerful model for more accurate embeddings at up to 15x increased compute times
SENTENCE_TRANSFORMER_MODEL = SENTENCE_TRANSFORMER_MODEL_FAST

c:\Users\mruss\projects\classes\ds587\ai-sycophancy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
responses = get_all_responses()
prompts = get_all_prompts()
system_prompts = get_all_system_prompts()
# YTA only for syccophancy analysis set
yta_prompts = prompts[prompts["YTA_NTA"] == "YTA"]
# Drop records where the LLM didn't make a firm YTA/NTA call
valid_responses = responses.dropna(subset="llm_label")

##### CSV Read-Write Path Config

In [3]:
embeddings_dir = Path().resolve() / "datasets" / "embeddings"
embeddings_dir.mkdir(exist_ok=True, parents=True)

prompt_emb_path = embeddings_dir / "prompt_embeddings.csv"
crowd_emb_path = embeddings_dir / "crowd_response_embeddings.csv"
llm_emb_path = embeddings_dir / "llm_response_embeddings.csv"

**Initial Setup**

To compare the semantic similarity between prompts and responses, we'll begin by using SBERT(**EXPLANATION**) to embed the meanings of all prompts and model responses into 768 vectors.

In [21]:
model = SentenceTransformer(SENTENCE_TRANSFORMER_MODEL)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2221.16it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
prompt_embeddings = model.encode(yta_prompts.prompt.tolist(), show_progress_bar=True)

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches: 100%|██████████| 63/63 [01:10<00:00,  1.12s/it]


In [23]:
llm_response_embeddings = model.encode(
    valid_responses.response.tolist(), show_progress_bar=True
)

Batches: 100%|██████████| 656/656 [04:30<00:00,  2.43it/s]


In [24]:
crowd_response_embeddings = model.encode(
    yta_prompts.top_comment.tolist(), show_progress_bar=True
)

Batches: 100%|██████████| 63/63 [00:25<00:00,  2.49it/s]


We'll also give some helpers for reading and writing these embeddings to resume runs later

In [8]:
def embeddings_to_csv(
    row_ids: Sequence, embeddings: torch.Tensor | np.ndarray, path: Path
):
    """Save row_id + embeddings to a CSV file

    Note: Embeddings are occassionally returned as pytorch tensors and occassionally returned as numpy arrays"""
    if embeddings.ndim != 2:
        raise ValueError(f"Expected a 2D Tensor got shape {embeddings.shape}")

    n_rows, n_cols = embeddings.shape
    if len(row_ids) != n_rows:
        raise ValueError("row_id length must match number of embedding rows")

    if isinstance(embeddings, np.ndarray):
        emb_arr = embeddings
    elif isinstance(embeddings, torch.Tensor):
        emb_arr = embeddings.detach().cpu().numpy()
    else:
        raise TypeError(f"Expected either Tensor or np.ndarray, got {type(embeddings)}")

    emb_df = pd.DataFrame(emb_arr)
    emb_df.columns = [f"emb_{i}" for i in range(n_cols)]

    df = pd.DataFrame({"row_id": list(row_ids)})
    df = pd.concat((df, emb_df), axis=1)

    df.to_csv(path, index = False)
    print(f"Wrote {len(df):,} embedding records to {path}")


def csv_to_embeddings(
    path: Path, keep_ids: set | None = None, dtype: torch.dtype = torch.float32
) -> tuple[pd.DataFrame, torch.Tensor]:
    df = pd.read_csv(path)
        
    if "row_id" not in df.columns:
        raise ValueError("CSV must contain a 'row_id' column")

    if keep_ids is not None:
        df = df[df['row_id'].isin(keep_ids)]
    
    emb_columns = [c for c in df.columns if c.startswith("emb_")]
    if not emb_columns:
        raise ValueError("No embedding columns found")
    embeddings = torch.tensor(df[emb_columns].to_numpy(), dtype=dtype)

    return df, embeddings

In [26]:
embeddings_to_csv(yta_prompts.prompt_id, prompt_embeddings, prompt_emb_path)
embeddings_to_csv(yta_prompts.prompt_id, crowd_response_embeddings, crowd_emb_path)
# NOTE: This embedding is in a different dimension that the other two
# the first two have one record for each PROMPT (N), this has one record for each valid MODEL response = |M| * N
embeddings_to_csv(valid_responses.response_id, llm_response_embeddings, llm_emb_path)

Wrote 2,000 embedding records to C:\Users\mruss\projects\classes\ds587\ai-sycophancy\datasets\embeddings\prompt_embeddings.csv
Wrote 2,000 embedding records to C:\Users\mruss\projects\classes\ds587\ai-sycophancy\datasets\embeddings\crowd_response_embeddings.csv
Wrote 20,968 embedding records to C:\Users\mruss\projects\classes\ds587\ai-sycophancy\datasets\embeddings\llm_response_embeddings.csv


## Analyzing Prompt Embedding Similarity 

Parition the prompts for LLM sycophancy label from each model 

For each model, $m$ define the sycophantic (+) and nonsycophantic subsets (-): 
$$
P_{m,+}=\set{i:y_i^m=1}, P_{m,-} = \set{i:y_i^m=0}
$$ 

In [14]:
# ARBTIRARILY FIX A MODEL AND SYSTEM PROMPT. WE'LL REVISIT
from models import Model
from prompts import SystemPrompt
from dataclasses import dataclass

@dataclass
class FilterConfig:
    system_prompt_id: int 
    system_prompt: SystemPrompt
    model: Model

@dataclass
class Subsets:
    syccophantic: torch.Tensor
    non_syccophantic: torch.Tensor

def get_filtered_responses(config: FilterConfig) -> pd.DataFrame:
    global responses
    filtered_responses = responses[
        (responses.model == config.model) & 
        (responses.system_prompt_id == config.system_prompt_id)
    ]
    return filtered_responses

def get_response_subsets(config: FilterConfig) -> Subsets:
    df = get_filtered_responses(config)
    syccophantic = df[df.llm_label == 'NTA']
    non_syccophantic = df[df.llm_label == 'YTA']
    
    return Subsets(
        csv_to_embeddings(llm_emb_path, keep_ids = set(syccophantic.prompt_id))[1],
        csv_to_embeddings(llm_emb_path, keep_ids = set(non_syccophantic.prompt_id))[1]
    )

In [16]:
cfg = FilterConfig(
    system_prompt_id=1,
    system_prompt= SystemPrompt.BASE,
    model = Model.GEMINI,
)

subsets = get_response_subsets(cfg)

### Prompt-Side Evaluation Define the within-group similarity sets (positive and negative) 

We define the positive and negative similarity subsets 

$$
S_{m,+} = \set{\cos(e_i^p, e_j^p): i, j \in P_{m, +}, i < j}\\ 
S_{m,-} = \set{\cos(e_i^p, e_j^p): i, j \in P_{m, -}, i < j} 
$$ 

Define the Between Group Similarity set 

$$
S_{m, \pm} = \set{\cos(e_i^p, e_j^p): i \in P_{m, +}, j \in P_{m, -}}
$$ 


In [18]:
import torch.nn.functional as F

def compute_similarity_subset(embeddings: torch.Tensor) -> torch.Tensor:
    """
    embeddings: [n, d]
    returns: [n * (n - 1) // 2] tensor of pairwise cosine similarities
             for all pairs i < j
    """
    # Normalize each embedding to unit length
    emb = F.normalize(embeddings, p=2, dim=1)

    # Full cosine similarity matrix: [n, n]
    sim_matrix = emb @ emb.T

    # Extract upper triangle, excluding diagonal (i < j)
    i, j = torch.triu_indices(sim_matrix.size(0), sim_matrix.size(1), offset=1)
    return sim_matrix[i, j]

def compute_between_group_subset(subsets: Subsets) -> torch.Tensor:
    """
    Computes S_{m,±}: cosine similarities for all pairs
    (i, j) where i ∈ P_{m,+}, j ∈ P_{m,-}.

    Returns a flattened tensor of shape [n+ * n-].
    """
    emb_pos = F.normalize(subsets.syccophantic, p=2, dim=1)  # [n+, d]
    emb_neg = F.normalize(subsets.non_syccophantic, p=2, dim=1)  # [n-, d]

    # Cross-group cosine similarity matrix: [n+, n-]
    # Every (i, j) entry is a valid between-group pair
    cross_sim = emb_pos @ emb_neg.T

    return cross_sim.flatten()  # [n+ * n-]

positive_sim_subset = compute_similarity_subset(subsets.syccophantic)
negative_sim_subset = compute_similarity_subset(subsets.non_syccophantic)
between_group_subset = compute_between_group_subset(subsets)

Then, we calculate the corresponding mean similarities:

$$ 
C_{m, +} = \frac1{|S_{m,+}|}\sum_{s\in S_{m, +}}s\\
C_{m, -} = \frac1{|S_{m,-}|}\sum_{s\in S_{m, -}}s\\ 
B_m = \frac1{|S_{m,\pm}|}\sum_{s\in S_{m, \pm}}s
$$ 

In [19]:
positive_sim_mean = positive_sim_subset.mean()
negative_sim_mean = negative_sim_subset.mean()
between_group_mean = between_group_subset.mean()

Then Report Two prompt-side metrics 

#### **Relative Cohesion** 

$$
\Delta \text{prompt}(m) = C_{m, +} - C{m, -}
$$ 

**Interpretation**: 
- $>0$: prompts that produce sycophantic responses from model m are more similar to each other than prompts that produce non-sycophantic responses 
- $≈0$: no real difference in prompt cohesion 
- $<0$ : sycophancy-inducing prompts are less semantically cohesive

#### **Positive-Group Similarity** 

$$
\Gamma \text{prompt}(m) = C_{m, +} - B_m
$$ 

**Interpretation** 
- $>0$: positive prompts are logcally distinct from negatives 
- $\approx 0$: positives are not more similar to one another than to negatives 
- $<0$: no evidence of separability 

In [20]:
def calculate_prompt_metrics(
    C_m_plus: torch.Tensor,
    C_m_minus: torch.Tensor,
    B_m: torch.Tensor,
    model_name: str = "m",
    system_prompt_name: str = "",
    verbose: bool = False
) -> dict:
    delta = (C_m_plus - C_m_minus).item()
    gamma = (C_m_plus - B_m).item()

    # Relative Cohesion interpretation
    if delta > 0.01:
        delta_interp = "sycophancy-inducing prompts are more semantically cohesive than non-sycophantic prompts"
    elif delta < -0.01:
        delta_interp = "sycophancy-inducing prompts are less semantically cohesive than non-sycophantic prompts"
    else:
        delta_interp = "no meaningful difference in prompt cohesion between groups"

    # Positive-Group Separability interpretation
    if gamma > 0.01:
        gamma_interp = "sycophancy-inducing prompts are more similar to each other than to non-sycophantic prompts — positive group is separable"
    elif gamma < -0.01:
        gamma_interp = "no evidence of separability — positive prompts are not closer to each other than to negatives"
    else:
        gamma_interp = "positive prompts are not meaningfully more similar to one another than to negatives"

    if verbose:
        print(f"=== Prompt-Side Metrics for model: {model_name} and system prompt {system_prompt_name}===\n")
        print(f"  C_m+  (mean within sycophantic):     {C_m_plus.item():.4f}")
        print(f"  C_m-  (mean within non-sycophantic): {C_m_minus.item():.4f}")
        print(f"  B_m   (mean between groups):          {B_m.item():.4f}")
        print()
        print(f"  Δprompt(m) = C_m+ - C_m-  = {delta:+.4f}")
        print(f"  → {delta_interp}")
        print()
        print(f"  Γprompt(m) = C_m+ - B_m   = {gamma:+.4f}")
        print(f"  → {gamma_interp}")
    
    return {
        'model_name': model_name,
        'system_prompt': system_prompt_name, 
        'delta': delta,
        'gamma': gamma,
        'delta_interp': delta_interp,
        'gamma_interp': gamma_interp
    }

Now, we iteratively calculate this for all Models and System Prompts!

In [24]:
def calculate_metrics_pipeline(config: FilterConfig, verbose: bool = True) -> dict:
    subsets = get_response_subsets(config)
    positive_sim_subset = compute_similarity_subset(subsets.syccophantic)


    negative_sim_subset = compute_similarity_subset(subsets.non_syccophantic)
    between_group_subset = compute_between_group_subset(subsets)
    positive_sim_mean = positive_sim_subset.mean()
    negative_sim_mean = negative_sim_subset.mean()
    between_group_mean = between_group_subset.mean()
    
    return calculate_prompt_metrics(
        positive_sim_mean, 
        negative_sim_mean, 
        between_group_mean, 
        model_name = config.model.value,
        system_prompt_name = config.system_prompt.name,
        verbose = verbose
    )

def calculate_all_model_semantic_metrics(verbose: bool = False) -> pd.DataFrame:
    records = []
    for model in Model:
        # Start indexing at one
        for (system_prompt_id, system_prompt) in enumerate(SystemPrompt, 1):
            config = FilterConfig(system_prompt_id, system_prompt, model)
            results = calculate_metrics_pipeline(config, verbose)
            records.append(results)
            print(f'Finishing processing model {model.value} with sys prompt {system_prompt.name}.')
    
    return pd.DataFrame(records)

In [25]:
all_model_prompt_semantic_metrics = calculate_all_model_semantic_metrics()

Finishing processing model claude-haiku-4-5-20251001 with sys prompt BASE.
Finishing processing model claude-haiku-4-5-20251001 with sys prompt HONEST_ASSISTANT.
Finishing processing model claude-haiku-4-5-20251001 with sys prompt THERAPY_FOCUSED.
Finishing processing model gpt-4.1-mini with sys prompt BASE.
Finishing processing model gpt-4.1-mini with sys prompt HONEST_ASSISTANT.
Finishing processing model gpt-4.1-mini with sys prompt THERAPY_FOCUSED.
Finishing processing model gpt-5.4-mini with sys prompt BASE.
Finishing processing model gpt-5.4-mini with sys prompt HONEST_ASSISTANT.
Finishing processing model gpt-5.4-mini with sys prompt THERAPY_FOCUSED.
Finishing processing model gemini-2.5-flash with sys prompt BASE.
Finishing processing model gemini-2.5-flash with sys prompt HONEST_ASSISTANT.
Finishing processing model gemini-2.5-flash with sys prompt THERAPY_FOCUSED.


In [26]:
all_model_prompt_semantic_metrics

,model_name,system_prompt,delta,gamma,delta_interp,gamma_interp
0,claude-haiku-4-5-20251001,BASE,-0.002603,-0.001320,no meaningful difference in prompt cohesion be...,positive prompts are not meaningfully more sim...
1,claude-haiku-4-5-20251001,HONEST_ASSISTANT,-0.006112,-0.003230,no meaningful difference in prompt cohesion be...,positive prompts are not meaningfully more sim...
2,claude-haiku-4-5-20251001,THERAPY_FOCUSED,-0.000165,-0.000192,no meaningful difference in prompt cohesion be...,positive prompts are not meaningfully more sim...
3,gpt-4.1-mini,BASE,-0.005949,-0.003206,no meaningful difference in prompt cohesion be...,positive prompts are not meaningfully more sim...
4,gpt-4.1-mini,HONEST_ASSISTANT,0.000278,0.000058,no meaningful difference in prompt cohesion be...,positive prompts are not meaningfully more sim...
5,gpt-4.1-mini,THERAPY_FOCUSED,-0.000865,-0.000627,no meaningful difference in prompt cohesion be...,positive prompts are not meaningfully more sim...
6,gpt-5.4-mini,BASE,0.006100,0.002958,no meaningful difference in prompt cohesion be...,positive prompts are not meaningfully more sim...
7,gpt-5.4-mini,HONEST_ASSISTANT,NaN,NaN,no meaningful difference in prompt cohesion be...,positive prompts are not meaningfully more sim...
8,gpt-5.4-mini,THERAPY_FOCUSED,-0.003962,-0.002071,no meaningful difference in prompt cohesion be...,positive prompts are not meaningfully more sim...
9,gemini-2.5-flash,BASE,0.009096,0.004721,no meaningful difference in prompt cohesion be...,positive prompts are not meaningfully more sim...



### Optional exploratory analyses 

To assess whether prompt embeddings could be useful for detection, optionally compute:
- k-nearest-neighbor purity on prompt embeddings by model-specific label 
- PCA visualizations of prompt embeddings colored by $y_im$ 
- Bootstrap cofidence intervals for each $\Gamma (m)$ then visualize 

These are exploratory and should be presented as supporting evidence rather than core inferential results.

In [ ]:
# P1 Exploratory Code Here

## LLM-to-Crowd Response Divergence

Partition the prompt items by the model-specific sycophancy label for each model.

For each model, $m$ define the sycophantic (+) and nonsycophantic subsets (-):

$$P_{m,+}=\set{i:y_{im}=1}, \, P_{m,-}=\set{i:y_{im}=0}$$

For each prompt item $i$ and model $m$:

- the model response is $r_{im}$
- the most upvoted crowd response is $r_i^C$

We treat the most upvoted crowd response $r_i^C$ as the reference human response for prompt $i$.

> *NOTE*: This should all largely overlap with Part 1 work. Will be stored in a CSV

### SBERT Crowd Divergence

Define the SBERT-based crowd divergence for each prompt-model pair:

$$d_{im}^{SBERT}=1-\cos(\text{SBERT}(r_{im}), \text{SBERT}(r_i^C))$$

Here:

- $d_{im}^{SBERT}\approx 0$: the model response is very close to the crowd-endorsed response
- larger values of $d_{im}^{SBERT}$ mean the model response deviates more from the crowd-endorsed response

Define the corresponding mean crowd divergence within each group:

$$
D_{m,+}^{SBERT}=\frac{1}{|P_{m,+}|}\sum_{i\in P_{m,+}} d_{im}^{SBERT}\\
D_{m,-}^{SBERT}=\frac{1}{|P_{m,-}|}\sum_{i\in P_{m,-}} d_{im}^{SBERT}
$$

Then define the model-specific excess crowd divergence:

$$\Gamma_{\text{crowd}}^{SBERT}(m)=D_{m,+}^{SBERT}-D_{m,-}^{SBERT}$$

**Interpretation**:

- $>0$: responses labeled sycophantic diverge more from the crowd-endorsed response than responses labeled non-sycophantic
- $\approx 0$: little or no difference in crowd divergence between the two groups
- $<0$: responses labeled sycophantic diverge less from the crowd-endorsed response than responses labeled non-sycophantic

### BERTScore Robustness Check

As a secondary robustness check, define the BERTScore-based crowd divergence for each prompt-model pair:

$$d_{im}^{BERT}=1-\text{BERTScore-F1}(r_{im}, r_i^C)$$

Define the corresponding mean BERTScore crowd divergence within each group:

$$
D_{m,+}^{BERT}=\frac{1}{|P_{m,+}|}\sum_{i\in P_{m,+}} d_{im}^{BERT}\\
D_{m,-}^{BERT}=\frac{1}{|P_{m,-}|}\sum_{i\in P_{m,-}} d_{im}^{BERT}
$$

Then define the model-specific excess crowd divergence under BERTScore:

$$\Gamma_{\text{crowd}}^{BERT}(m)=D_{m,+}^{BERT}-D_{m,-}^{BERT}$$

**Interpretation**:

- $>0$: responses labeled sycophantic diverge more from the crowd-endorsed response under BERTScore
- $\approx 0$: little or no difference in divergence under BERTScore
- $<0$: responses labeled sycophantic diverge less from the crowd-endorsed response under BERTScore

### Recommended reporting

For each model $m$, report:

- $D_{m,+}^{SBERT}$
- $D_{m,-}^{SBERT}$
- $\Gamma_{\text{crowd}}^{SBERT}(m)$

and optionally:

- $D_{m,+}^{BERT}$
- $D_{m,-}^{BERT}$
- $\Gamma_{\text{crowd}}^{BERT}(m)$

### Optional exploratory analyses

To assess whether sycophancy is associated with large crowd divergence patterns across models, optionally compute:

- boxplots or violin plots of $d_{im}^{SBERT}$ by $y_{im}$ for each model
- bootstrap confidence intervals for each $\Gamma_{\text{crowd}}^{SBERT}(m)$
- the same visual summaries for BERTScore as a robustness check

These are exploratory and should be presented as supporting evidence rather than core inferential results.

In [ ]:
# Part Two Code Here

### Extra Questions

- Is there a relationship between prompt length and model response (simple lin reg)
